# Colab Fine-Tuning: MiniLM Pretrained on Pilot (Khung 1 - Hiếu)

**Owner:** Hiếu (Person 3)  
**Objective:** Fine-tune `sentence-transformers/all-MiniLM-L6-v2` with `SentenceTransformerTrainer` + `MultipleNegativesRankingLoss`, batch sampler `NO_DUPLICATES`, evaluate every epoch with `InformationRetrievalEvaluator` on validation queries, demonstrate checkpoint save & resume, and export for local CPU verification.

> **Runtime:** `Runtime > Change runtime type > T4 GPU` (free Colab GPU). A CPU pathway with small steps is also provided.

## 1. Bootstrap & Hardware Setup

In [ ]:
# 1. Clone repo or pull latest branch
import os, subprocess, platform, torch

REPO_URL = "https://github.com/minhhoangb/food-recipe-recommendation-system.git"
BRANCH = "feature/person3-finetune-training"

if not os.path.exists("/content/repo"):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, "/content/repo"], check=True)
os.chdir("/content/repo")

# 2. Install dependencies
!pip install -q -r requirements.txt

# 3. Check real hardware (DoD: ghi đúng GPU thật, không bịa)
print("Platform:", platform.platform())
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU mode (small steps pathway enabled).")

In [ ]:
# 4. Optional: Mount Google Drive for persistent checkpointing
checkpoint_dir = "./checkpoints"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    checkpoint_dir = "/content/drive/MyDrive/food_recipe_checkpoints"
    os.makedirs(checkpoint_dir, exist_ok=True)
    print(f"Using Google Drive checkpoint dir: {checkpoint_dir}")
except Exception:
    os.makedirs(checkpoint_dir, exist_ok=True)
    print(f"Using local checkpoint dir: {checkpoint_dir}")

## 2. Build Training Pairs & Inspect Manifest (DoD a)

- Uses `src.build_pairs` to generate query-positive pairs from `train` split only.
- Query created with `src.queries.make_query_ingredients` using diverse seeds.
- Positive formatted via `src.embeddings.format_document` (Hưng's E04 format).
- Max 10,000 pairs, seed 42. Writes manifest with verified sha256.

In [ ]:
# Generate pairs (pilot corpus, max 10,000 pairs, seed 42)
!python -m src.build_pairs --config configs/pilot.yaml --split train --mode ingredients --max-pairs 10000 --seed 42

# Display manifest with hashes
import json
from pathlib import Path
manifest_path = Path("data/pairs/pilot_train_ingredients_pairs_manifest.json")
if manifest_path.exists():
    print("=== Pairs Manifest ===")
    print(json.dumps(json.loads(manifest_path.read_text()), indent=2))

## 3. Prepare Datasets & InformationRetrievalEvaluator (DoD b)

Prepares training dataset and validation evaluator using queries, corpus, and qrels from `data/processed/pilot/`.

In [ ]:
from datasets import Dataset
from src.build_pairs import load_pairs
from src.utils import read_jsonl
from sentence_transformers.evaluation import InformationRetrievalEvaluator

# Load training pairs
pairs_data = load_pairs("data/pairs/pilot_train_ingredients_pairs.jsonl")
train_dataset = Dataset.from_dict({
    "query": [p["query"] for p in pairs_data],
    "positive": [p["positive"] for p in pairs_data],
})
print(f"Loaded {len(train_dataset)} training pairs for Trainer.")

# Prepare validation evaluator
val_queries_raw = [r for r in read_jsonl("data/processed/pilot/queries.jsonl") if r.get("split") == "validation"]
val_corpus_raw = list(read_jsonl("data/processed/pilot/corpus.jsonl"))
val_qrels_raw = list(read_jsonl("data/processed/pilot/qrels.jsonl"))

val_queries = {q["query_id"]: ", ".join(q["ingredients"]) for q in val_queries_raw}
val_corpus = {c["recipe_id"]: ", ".join(c["ingredients_normalized"]) for c in val_corpus_raw}
val_relevant_docs = {}
for row in val_qrels_raw:
    if row["query_id"] in val_queries and int(row["relevance"]) >= 1:
        val_relevant_docs.setdefault(row["query_id"], set()).add(row["recipe_id"])

evaluator = InformationRetrievalEvaluator(
    queries=val_queries,
    corpus=val_corpus,
    relevant_docs=val_relevant_docs,
    name="validation_pilot",
    mrr_at_k=[10],
    ndcg_at_k=[10],
    accuracy_at_k=[5, 10],
    show_progress_bar=False,
)
print(f"Evaluator initialized with {len(val_queries)} validation queries over {len(val_corpus)} corpus recipes.")

## 4. Pre-training Baseline Evaluation (MRR@10 before train)

In [ ]:
from sentence_transformers import SentenceTransformer

base_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda" if torch.cuda.is_available() else "cpu")
baseline_metrics = evaluator(base_model)

pre_train_mrr = baseline_metrics.get("validation_pilot_cosine_mrr@10", 0.0)
print(f"Baseline (pre-training) Validation MRR@10: {pre_train_mrr:.4f}")

## 5. Fine-Tuning with SentenceTransformerTrainer & MultipleNegativesRankingLoss (DoD b & c)

- Model: `all-MiniLM-L6-v2`, `max_seq_length=128`
- Loss: `MultipleNegativesRankingLoss`
- Batch sampler: `BatchSamplers.NO_DUPLICATES`
- Evaluator: evaluated every epoch
- Checkpointing: `save_steps=50`, `resume_from_checkpoint` demonstrated.

In [ ]:
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments, losses
from sentence_transformers.training_args import BatchSamplers

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
model.max_seq_length = 128
loss = losses.MultipleNegativesRankingLoss(model)

is_cuda = torch.cuda.is_available()
batch_size = 32 if is_cuda else 16
epochs = 1

training_args = SentenceTransformerTrainingArguments(
    output_dir=checkpoint_dir,
    num_train_epochs=epochs,
    per_device_train_batch_size=batch_size,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=3,
    eval_strategy="epoch",
    logging_steps=10,
    # On CPU, limit to 10 steps for fast sanity check
    max_steps=10 if not is_cuda else -1,
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=None,
    loss=loss,
    evaluator=evaluator,
)

# Start training
print("Starting training with SentenceTransformerTrainer...")
train_result = trainer.train()
print("Training finished. Loss history recorded.")

## 6. Checkpoint & Resume Verification (DoD c)

Demonstrates resuming from the latest saved checkpoint, proving steps and loss continuity.

In [ ]:
# Find latest checkpoint directory
from transformers.trainer_utils import get_last_checkpoint
last_ckpt = get_last_checkpoint(checkpoint_dir)
print(f"Latest checkpoint found: {last_ckpt}")

if last_ckpt:
    print("Resuming training from checkpoint to demonstrate continuity...")
    resume_result = trainer.train(resume_from_checkpoint=last_ckpt)
    print("Resume completed successfully!")

## 7. Post-Training Evaluation (MRR@10 after train)

In [ ]:
post_metrics = evaluator(model)
post_train_mrr = post_metrics.get("validation_pilot_cosine_mrr@10", 0.0)

print("=========================================")
print(f"Before Fine-Tuning MRR@10 : {pre_train_mrr:.4f}")
print(f"After Fine-Tuning MRR@10  : {post_train_mrr:.4f}")
print(f"Improvement (Delta)       : {post_train_mrr - pre_train_mrr:+.4f}")
print("=========================================")

## 8. Export Model & Local CPU Top-K Equivalence Test (DoD d)

- Save model checkpoint via `save_pretrained`.
- Re-load on local CPU.
- Encode 20 sample queries and 200 documents, assert ranking equivalence with Colab.

In [ ]:
export_path = "models/minilm_finetuned_checkpoint"
model.save_pretrained(export_path)
print(f"Model exported successfully to: {export_path}")

# Reload on CPU to verify export
cpu_model = SentenceTransformer(export_path, device="cpu")
print("Successfully reloaded exported model on CPU!")

# Equivalence check: 20 queries, 200 docs
import numpy as np
sample_q_texts = list(val_queries.values())[:20]
sample_doc_texts = list(val_corpus.values())[:200]

colab_q_vecs = model.encode(sample_q_texts, normalize_embeddings=True)
cpu_q_vecs = cpu_model.encode(sample_q_texts, normalize_embeddings=True)

colab_doc_vecs = model.encode(sample_doc_texts, normalize_embeddings=True)
cpu_doc_vecs = cpu_model.encode(sample_doc_texts, normalize_embeddings=True)

# Cosine similarities
colab_sims = np.dot(colab_q_vecs, colab_doc_vecs.T)
cpu_sims = np.dot(cpu_q_vecs, cpu_doc_vecs.T)

max_diff = np.max(np.abs(colab_sims - cpu_sims))
print(f"Max absolute difference between Colab and CPU embeddings: {max_diff:.6f}")
assert max_diff < 1e-4, "CPU and Colab representations differ by more than allowed tolerance!"

# Compare top-k
colab_top5 = np.argsort(-colab_sims, axis=1)[:, :5]
cpu_top5 = np.argsort(-cpu_sims, axis=1)[:, :5]
matches = np.mean(colab_top5 == cpu_top5)
print(f"Top-5 ranking match agreement: {matches * 100:.1f}%")
print("Checkpoint validated for CPU inference and ready for Hưng via embeddings.py --model-path!")